In [1]:
import geopandas as gpd
import pandas as pd
import osmnx as ox
import matplotlib.pyplot as plt
import os
import folium
from folium.plugins import Fullscreen
import alphashape
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.animation as animation
import imageio
import networkx as nx
import io
from PIL import Image
import time
import sys
from matplotlib import cm
from IPython.display import clear_output
import seaborn as sns
import matplotlib.colors as mcolors
from shapely import box
import gc
pd.options.mode.copy_on_write = True
import scipy
%matplotlib inline
from shapely.geometry import Point
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from ipywidgets import interact, FloatSlider
from rasterstats import point_query
from datetime import datetime
from shortest_path_util import penalty_turns
from shortest_path_turn_penalty import shortest_path_turn_penalty
import shutil
import importlib
import shortest_path_turn_penalty,shortest_path_util
importlib.reload(shortest_path_turn_penalty)
importlib.reload(shortest_path_util)
from datetime import datetime

In [149]:
class Network:
    def __init__(self, crs = 32618):
        self.crs = crs
        self.n_pot = None
        self.n_pot_og = None
        self.n_pot_nodes = None
        self.n_pot_edges = None
        self.n_ex = None
        self.n_ex_og = None
        self.n_ex_nodes = None
        self.n_ex_edges = None
        self.trips = None
        self.map_pot = None
        self.map_ex = None
        self.map_routes = None
        self.trips = None
        self.trips_within = None
        self.sample = None
        self.boundaries = None
        self.routes = None
        self.unsolved_routes_ids = []
        self.all_routes_edges = None
        self.routes_summary = None
        self.evol = []
        self.associated_links = None
        self.transit_stops = None
        self.traffic_signals = None
        self.turn_penalties = None
        
    def load_n_pot(self,path):
        self.n_pot = ox.io.load_graphml(path)
        self.n_pot = ox.project_graph(self.n_pot)
        self.n_pot_nodes,self.n_pot_edges = ox.graph_to_gdfs(self.n_pot)
        used_nodes = set(self.n_pot_edges.index.get_level_values('u')) | set(self.n_pot_edges.index.get_level_values('v'))
        self.n_pot_nodes = self.n_pot_nodes[self.n_pot_nodes.index.isin(used_nodes)]
        self.n_pot_edges['build_iter'] = 0
        self.n_pot_og = ox.graph_from_gdfs(self.n_pot_nodes.copy(),self.n_pot_edges.copy())
        
    def filter_n_pot(self):
        self.n_pot_edges[self.n_pot_eges.highway.isin(['residential','primary','secondary','tertiary','tertiary_link'])]
        self.n_pot = ox.graph_from_gdfs(self.n_pot_nodes,self.n_pot_edges)

    def plot_n_pot(self, nodes = False):
        if self.n_pot == None:
            print('No potential network loaded')
        else:
            self.map_pot = self.n_pot_edges.explore(color = 'black')
            if nodes:
                self.n_pot_nodes.explore(m = self.map_pot)
            display(self.map_pot)

    def load_n_ex(self,path):
        self.n_ex = ox.io.load_graphml(path)
        self.n_ex = ox.project_graph(self.n_ex)
        self.n_ex_nodes,self.n_ex_edges = ox.graph_to_gdfs(self.n_ex)
        used_nodes = set(self.n_ex_edges.index.get_level_values('u')) | set(self.n_ex_edges.index.get_level_values('v'))
        self.n_ex_nodes = self.n_ex_nodes[self.n_ex_nodes.index.isin(used_nodes)]
        self.n_ex_edges['build_iter'] = 0
        self.n_ex_og = ox.graph_from_gdfs(self.n_ex_nodes.copy(),self.n_ex_edges.copy())

    def filter_n_ex(self,highway_type = 'cycleway'):
        self.n_ex_edges = self.n_ex_edges[self.n_ex_edges.highway == highway_type]
        self.n_ex = ox.graph_from_gdfs(self.n_ex_nodes,self.n_ex_edges)

    def plot_n_ex(self, nodes = False):
        if self.n_ex == None:
            print('No existing network loaded')
        else:
            self.map_ex = self.n_ex_edges.explore(color = 'black')
            if nodes:
                self.n_ex_nodes.explore(m = self.map_ex)
            display(self.map_ex)

    def load_trips(self, path, source_crs, delimiter = ',',cols = ['ipere','mode','motif','age','xdomi','ydomi','faclog_s27','xorig','yorig','xdest','ydest','potVelo','fexpPotVelo', 'velo'],potvel = True):
        self.trips = pd.read_csv(path,delimiter = delimiter)
        if potvel == True:
            self.trips = self.trips[self.trips.potVelo == 1]
        else:
            self.trips = self.trips[(self.trips.potVelo == 1)|(self.trips.velo == 'X')]
        self.trips = self.trips[cols]
        self.trips = gpd.GeoDataFrame(self.trips, geometry=gpd.points_from_xy(self.trips.xorig, self.trips.yorig), crs=source_crs)
        self.trips = self.trips.rename(columns = {'geometry': 'orig'})
        self.trips = self.trips.assign(dest = gpd.points_from_xy(self.trips.xdest, self.trips.ydest,crs=source_crs))
        # self.trips = self.trips.assign(domi = gpd.points_from_xy(self.trips.xdomi, self.trips.ydomi,crs=source_crs))
        self.trips = self.trips.set_geometry('orig')
        self.trips = self.trips.set_crs(source_crs)
        self.trips = self.trips.to_crs(self.crs)
        self.boundaries = gpd.GeoDataFrame({'name':['region']},geometry = gpd.GeoSeries(self.n_pot_edges.geometry.union_all()).concave_hull(0.2)
                                           , crs = self.crs)
        
        self.trips = self.trips.to_crs(self.crs)
        self.trips.index+=1
        self.trips_within = gpd.sjoin(self.trips, self.boundaries, how='inner', predicate='within')[list(self.trips.columns)]
        self.trips_within = self.trips_within.set_geometry('dest')
        self.trips_within = self.trips_within.to_crs(self.crs)
        self.trips_within = gpd.sjoin(self.trips_within, self.boundaries, how='inner', predicate='within')[list(self.trips.columns)]
        


    def sample_trips(self,sample_size = None, spec_route = None):
        if sample_size is None:
            sample_size = len(self.trips_within)
            self.sample = self.trips_within.sample(sample_size)
        else:
            self.sample = pd.concat([self.sample,self.trips_within.sample(sample_size)])
        if spec_route is not None:
            self.sample = self.trips_within[self.trips_within.ipere == spec_route]

    def compute_routes(self, network, weight = 'gencost'):
        network = ox.projection.project_graph(network, to_crs = self.crs)
        o_nodes,o_dists = ox.nearest_nodes(network,self.sample.orig.x.values,self.sample.orig.y.values, return_dist=True)
        d_nodes,d_dists = ox.nearest_nodes(network,self.sample.dest.x.values,self.sample.dest.y.values, return_dist=True)
        routes = ox.shortest_path(network, o_nodes, d_nodes, weight=weight,cpus = None)
        self.routes = pd.DataFrame(data = routes, index = self.sample.index)

    
    def reset_sample(self):
        self.sample = None
    def reset_routes(self):
        self.routes = []
        self.unsolved_routes_ids = []
        self.all_routes_edges = None
        self.routes_summary = None
        
    def get_routes_edges(self,network):
        route_edges = []
        static_route_ids = []
        for i in tqdm(range(len(self.routes)),leave = False):
            if self.routes.iloc[i].nodes is not None:
                if len(self.routes.iloc[i].nodes)>1:
                    edges = ox.routing.route_to_gdf(network,self.routes.iloc[i]['nodes'])
                    edges['route_number'] = self.sample.index[i]
                    route_edges.append(edges) 
                else:
                    static_route_ids.append(self.sample.index[i])
            
            else:
                self.unsolved_routes_ids.append(self.sample.index[i])
                continue
        print((1-len(self.unsolved_routes_ids)/len(self.sample))*100,'% of solved routes, ',len(static_route_ids)/len(self.sample)*100,
        ' % of static routes')
        self.sample = self.sample.drop(self.unsolved_routes_ids)
        self.sample = self.sample.drop(static_route_ids)
        self.all_routes_edges = pd.concat(route_edges)
    
    def compute_routes_summary(self):
        
        self.routes_summary = self.all_routes_edges[['length','gencost','route_number']].groupby(['route_number']).sum()
        self.routes_summary['length_cycleway'] = self.all_routes_edges.groupby('route_number').apply(
            lambda x: x.loc[x["highway"] == "cycleway", "length"].sum(),include_groups=False)
        self.routes_summary['length_street'] = self.all_routes_edges.groupby('route_number').apply(
            lambda x: x.loc[x["highway"] != "cycleway", "length"].sum(),include_groups=False)
        self.routes_summary['prop_cycleway'] = self.routes_summary['length_cycleway']/self.routes_summary['length']

        # links = self.all_routes_edges[self.all_routes_edges.highway !='cycleway'].drop_duplicates()
        # flows = self.all_routes_edges[self.all_routes_edges.highway !='cycleway'].groupby(by = ['length'], as_index = False).size()['size']
        # links['flow'] = flows.values
        # links['flux'] = flows.values*links.length

    def plot_routes(self,network):

        if self.all_routes_edges is None:
            self.get_routes_edges(network)
        

        self.sample = self.sample.set_geometry('orig')
        self.map_routes = self.sample.explore(color='blue', name='orig')
        self.sample = self.sample.set_geometry('dest')
        map2 = self.sample.explore(color='red', name='dest', m=self.map_routes)
        map3 = self.boundaries.explore(m = self.map_routes, name = 'boundaries', fill = False)
        map4 = self.n_ex_edges.explore(color='black', name='existing', m=self.map_routes,style_kwds={'opacity': 0.3})
        map5 = self.n_pot_edges.explore(color='black', name='potential', m=self.map_routes,style_kwds={'opacity': 0.3})
        self.all_routes_edges['route_n'] = self.all_routes_edges['route_number'].astype(str)
        map8 = self.all_routes_edges.explore(column = 'route_n',cmap = 'gist_rainbow', name = 'routes', m = self.map_routes, legend = False
                                            ,style_kwds={'weight': 5})
        folium.LayerControl().add_to(self.map_routes)
        display(self.map_routes)

    def weight_network(self,cycleway_reduc_factor=0.9):
        self.n_pot_edges['gencost'] = self.n_pot_edges["length"] * self.n_pot_edges["highway"].apply(
            lambda x: cycleway_reduc_factor if x == "cycleway" else 1)
        self.n_pot = ox.graph_from_gdfs(self.n_pot_nodes,self.n_pot_edges)


    def compute_routes_replace(self,network,idxs,weight = 'gencost'):
        network = ox.projection.project_graph(network, to_crs = self.crs)
        subsample = self.sample.loc[idxs]
        o_nodes,o_dists = ox.nearest_nodes(network,subsample.orig.x.values,subsample.orig.y.values, return_dist=True)
        d_nodes,d_dists = ox.nearest_nodes(network,subsample.dest.x.values,subsample.dest.y.values, return_dist=True)
        routes = ox.shortest_path(network, o_nodes, d_nodes, weight=weight)
        route_edges = []
        for i in range(len(idxs)):
                if routes[i] is not None:
                    if len(routes[i])>1:
                        edges = ox.routing.route_to_gdf(network,routes[i])
                        edges['route_number'] = idxs[i]
                        route_edges.append(edges)
        newroutes = pd.concat(route_edges)
        self.all_routes_edges = self.all_routes_edges[~self.all_routes_edges['route_number'].isin(idxs)]
        self.all_routes_edges = pd.concat([self.all_routes_edges,newroutes])
        
    def run_algo(self,n_iter = 1000, budget = 10000):
        self.reset_routes()
        self.compute_routes(self.n_pot)
        self.get_routes_edges(self.n_pot)
 
        for i in tqdm(range(n_iter)):
            clear_output(wait = True)
            self.compute_routes_summary()
            preconnected = self.routes_summary[(self.routes_summary['length_cycleway']>0)&(self.routes_summary['length_street']>0)]
            route_id_to_add = preconnected[preconnected['length_street'] == preconnected['length_street'].min()].index[0]
            print('adding ',route_id_to_add)
            route_edges = self.all_routes_edges[self.all_routes_edges.route_number == int(route_id_to_add)]
            route_edges_to_add = route_edges[route_edges.highway!='cycleway']
    
            edges_id_to_add = route_edges_to_add.index
            self.n_pot_edges.loc[edges_id_to_add,'highway']='cycleway'
            self.n_pot_edges.loc[edges_id_to_add,'build_iter'] = np.max(self.n_pot_edges.build_iter)+1
            self.n_ex_edges = self.n_pot_edges[self.n_pot_edges.highway == 'cycleway']
    
            self.weight_network()
            self.n_pot = ox.graph_from_gdfs(self.n_pot_nodes,self.n_pot_edges)
    
            recompute_polygon = route_edges.buffer(500).union_all()
            recompute_polygon = gpd.GeoDataFrame([recompute_polygon]).rename(columns={0:'geometry'}).set_geometry('geometry')
            recompute_polygon = recompute_polygon.set_crs(self.crs)
            
            o_int = recompute_polygon.sjoin(self.sample.set_geometry('orig')).index_right.tolist()
            d_int = recompute_polygon.sjoin(self.sample.set_geometry('dest')).index_right.tolist()
    
            recompute_idxs = list(set(o_int+d_int))

            m = recompute_polygon.explore()
            self.sample.loc[o_int].explore(m = m, color = 'yellow')
            display(m)
            break
    
            if len(recompute_idxs) == 0:
                continue
            else:
                print('recomputing ',len(recompute_idxs))
                self.compute_routes_replace(self.n_pot,recompute_idxs)

In [3]:
n = Network()
# ex = 'Data/Reseaux/ns_EX_MTL.graphml'
# pot = 'Data/Reseaux/ns_POT_MTL.graphml'
ex = 'Data/Reseaux/ns_EX_PVR.graphml'
pot = 'Data/Reseaux/ns_POT_PVR.graphml'
trips = 'Data/DonneesOuvertes2025_bixi/bixi_trips_by_station.csv'
stm_gtfs = 'Data/gtfs_stm/stops.txt'
signals_file = 'Data/feux-circulation.csv'

In [ ]:
n.load_n_pot(pot)
n.load_n_ex(ex)
n.load_trips(trips,source_crs=4326,cols=['ipere','xorig','yorig','xdest','ydest','ntrips','potVelo'])

In [ ]:
stops = pd.read_csv(stm_gtfs)
metros = stops[stops['location_type']==1]
metros = gpd.GeoDataFrame(metros,geometry = gpd.points_from_xy(metros.stop_lon,metros.stop_lat),crs = 4326).to_crs(n.crs)
n.transit_stops=metros

In [4]:
input_tif = "Data/Elevation/output_AW3D30.tif"
output_tif = "Data/Elevation/output_AW3D30_32618.tif"
output_tif_upsampled = "Data/Elevation/output_AW3D30_32618_us.tif"
dst_crs = "EPSG:32618"  # UTM Zone 18N

def convert_tif_crs(input_file,output_file,dst_crs):
    with rasterio.open(input_file) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)
        
        kwargs = src.meta.copy()
        kwargs.update({
            "crs": dst_crs,
            "transform": transform,
            "width": width,
            "height": height
        })

        with rasterio.open(output_file, "w", **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.lanczos)

                
def smooth_raster(input_tif, output_tif, scale=2):
    with rasterio.open(input_tif) as src:
        transform, width, height = calculate_default_transform(
            src.crs, src.crs, src.width, src.height, *src.bounds,
            dst_width=int(src.width * scale), dst_height=int(src.height * scale)
        )

        kwargs = src.meta.copy()
        kwargs.update({
            'height': height,
            'width': width,
            'transform': transform
        })

        with rasterio.open(output_tif, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=src.crs,
                    resampling=Resampling.cubic_spline
                )

convert_tif_crs(input_tif,output_tif,dst_crs)
smooth_raster(output_tif,output_tif_upsampled,scale = 2)


In [150]:
def funcToMethod(func, clas, method_name=None):
    setattr(clas, method_name or func.__name__, func)
def run_algo(self, budget,n_iter = 1000,plot = True):
    if np.max(self.n_pot_edges.build_iter) == 0:
        self.reset_routes()
        self.evol = []
        self.compute_routes(self.n_pot)
        self.get_routes_edges(self.n_pot)
        self.compute_routes_summary()
        self.evol.append(self.routes_summary.copy().prop_cycleway)
    while self.n_ex_edges[self.n_ex_edges.build_iter > 0].length.sum()/1000 < budget:
        if plot:
            if i==0:
                fig, ax = plt.subplots(figsize = (5,5))
                ax.axis('off')
        clear_output(wait = True)
        print(f'Iteration {self.n_ex_edges[self.n_ex_edges.build_iter > 0].length.sum()/1000}/'+str(budget))
        preconnected = self.routes_summary[(self.routes_summary['length_cycleway']>0)&(self.routes_summary['length_street']>0)]
        if len(preconnected) == 0 and i<len(self.sample):
            preconnected = self.routes_summary[(self.routes_summary['length_street']>0)]
        route_id_to_add = preconnected.sort_values(by = 'fpkm', ascending = False).index[0]
        print(self.routes_summary.loc[route_id_to_add])
        recompute_summary_idxs = preconnected.sort_values(by = 'fpkm', ascending = False).index[:len(self.routes_summary)//10+1].tolist()
        
        
        route_edges = self.all_routes_edges[self.all_routes_edges.route_number == int(route_id_to_add)]
        route_edges_to_add = route_edges[route_edges.highway!='cycleway']

        edges_id_to_add = route_edges_to_add.index
        self.n_pot_edges.loc[edges_id_to_add,'highway']='cycleway'
        self.n_pot_edges.loc[edges_id_to_add,'build_iter'] = np.max(self.n_pot_edges.build_iter)+1
        self.n_ex_edges = self.n_pot_edges[self.n_pot_edges.highway == 'cycleway']
        self.all_routes_edges.loc[edges_id_to_add,'highway']='cycleway'

        self.compute_routes_summary(subsample_idxs = recompute_summary_idxs)
        self.evol.append(self.routes_summary.copy()['prop_cycleway'])
        if plot:
            if i%5== 0:
                
                n.n_ex_edges[n.n_ex_edges.build_iter<=i].plot(column = 'build_iter',cmap = 'viridis',ax=ax)
                clear_output(wait = True)
                display(fig)
                print(f'Iteration {i}/{n_iter}')


def is_almost_subset(a, b, tolerance=0.1):
    if not a:
        return True
    common = a & b
    match_fraction = len(common) / len(a)
    return match_fraction >= (1 - tolerance)

def get_mask(x, all_routes, tolerance):
    return [
        (route != x) and (len(route) != 0) and is_almost_subset(route, x, tolerance=tolerance)
        for route in all_routes
    ]

def count_left_turns(edges):
    angles = np.diff(edges)%360
    n_left_turns = (angles<330)*(angles>150)
    return n_left_turns.sum()

def count_right_turns(edges):
    angles = np.diff(edges)%360
    n_right_turns = (angles<=150)*(angles>30)
    return n_right_turns.sum()
    
def compute_routes_summary(self, beta_age,mu_age,scale_age,beta_transit,beta_bikeway,alpha,weight = None, proximity_dist = 10, subsample_idxs = None):
        
    if subsample_idxs is None:
        subsample = self.all_routes_edges
        subsample_idxs = self.sample.index
        self.routes_summary = subsample[['length','gencost','route_number']].groupby(['route_number']).sum()
        self.routes_summary = pd.merge(self.routes_summary,self.sample,left_index=True,right_index=True,how='left')
    else:
        subsample = self.all_routes_edges[self.all_routes_edges.route_number.isin(subsample_idxs)]
        
    # subsample['flow'] = subsample.join(subsample.index.value_counts(), how = 'left')['count']
    grouped_route_links = subsample.groupby('route_number')
        
    # self.routes_summary.loc[subsample_idxs,'crossed_int']=[np.sum(
    #     self.n_pot_nodes.loc[route].street_count==4) for route in self.routes.loc[subsample_idxs,"nodes"]]
    # self.routes_summary.loc[subsample_idxs,'crossed_traffic_signals']=[np.sum(
    #     self.n_pot_nodes.loc[route].traffic_signals==True) for route in self.routes.loc[subsample_idxs,"nodes"]]
    
    # self.routes_summary.loc[subsample_idxs,'int_per_km'] = self.routes_summary.loc[subsample_idxs,'crossed_int']/self.routes_summary.loc[subsample_idxs,'length']*1000 
    # self.routes_summary.loc[subsample_idxs,'n_left_turns'] = grouped_route_links.bearing.agg(count_left_turns)
    # self.routes_summary.loc[subsample_idxs,'n_right_turns'] = grouped_route_links.bearing.agg(count_right_turns)
    # self.routes_summary.loc[subsample_idxs,'n_turns'] = self.routes_summary.loc[subsample_idxs,'n_right_turns']+self.routes_summary.loc[subsample_idxs,'n_left_turns']
    
    # self.routes_summary.loc[subsample_idxs,'n_left_turns_per_km'] = self.routes_summary.loc[subsample_idxs,'n_left_turns']/self.routes_summary.loc[subsample_idxs,'length']*1000 
    # self.routes_summary.loc[subsample_idxs,'n_right_turns_per_km'] = self.routes_summary.loc[subsample_idxs,'n_right_turns']/self.routes_summary.loc[subsample_idxs,'length']*1000
    # self.routes_summary.loc[subsample_idxs,'n_turns_per_km'] = self.routes_summary.loc[subsample_idxs,'n_turns']/self.routes_summary.loc[subsample_idxs,'length']*1000
    
    self.routes_summary.loc[subsample_idxs,'length_cycleway'] = grouped_route_links.apply(
        lambda x: x.loc[x["highway"] == "cycleway", "length"].sum(),include_groups=False)
    self.routes_summary.loc[subsample_idxs,'length_street'] =grouped_route_links.apply(
        lambda x: x.loc[x["highway"] != "cycleway", "length"].sum(),include_groups=False)
    self.routes_summary.loc[subsample_idxs,'links_id_to_complete'] = grouped_route_links.apply(
        lambda x: set(x.loc[x["highway"] != "cycleway"].index.tolist()), include_groups = False)
    self.routes_summary.loc[subsample_idxs,'prop_cycleway'] = (self.routes_summary.loc[subsample_idxs,'length_cycleway']/
                                                                (self.routes_summary.loc[subsample_idxs,'length_street']+
                                                                self.routes_summary.loc[subsample_idxs,'length_cycleway']))
    
    # self.routes_summary.loc[subsample_idxs,'n_completed_routes'] = self.routes_summary['links_id_to_complete'].apply(
    #     lambda x: sum(route.issubset(x) for route in self.routes_summary['links_id_to_complete'] if (route != x)&(len(route)!=0)))
    # # self.routes_summary.loc[subsample_idxs,'n_near_completed_routes'] = self.routes_summary['links_id_to_complete'].apply(
    # #     lambda x: sum(is_almost_subset(route,x,tolerance = alpha) for route in self.routes_summary['links_id_to_complete']
    # #                   if (route != x)&(len(route)!=0)))

    masks = self.routes_summary['links_id_to_complete'].apply(
    lambda x: get_mask(x, self.routes_summary['links_id_to_complete'], alpha)
    )

    self.routes_summary.loc[subsample_idxs, 'n_near_completed_routes'] = masks.apply(sum)
    self.routes_summary.loc[subsample_idxs, 'near_completed_routes'] = masks.apply(
    lambda m: self.routes_summary.loc[m].index
    )
    
    
    # self.routes_summary.loc[subsample_idxs,'fpkm'] = grouped_route_links.apply(
    #     lambda x: ((x.loc[x["highway"] != "cycleway", "flow"]*
    #                x.loc[x["highway"] != "cycleway", "length"]).sum()/x.loc[x["highway"] != "cycleway", "length"].sum())
    #     if x.loc[x["highway"] != "cycleway", "length"].sum() != 0 else 0 ,include_groups=False)
    # self.routes_summary.loc[subsample_idxs,'normalized_fpkm'] = self.routes_summary.loc[subsample_idxs,'fpkm']/self.routes_summary['fpkm'].sum()

    if not 'sl_dist' in self.routes_summary.columns:
        o_nodes_idxs = self.routes.loc[subsample_idxs,'nodes'].str[0]
        d_nodes_idxs = self.routes.loc[subsample_idxs,'nodes'].str[-1]
        self.routes_summary.loc[subsample_idxs,'sl_dist'] = self.n_pot_nodes.loc[o_nodes_idxs].distance(self.n_pot_nodes.loc[d_nodes_idxs],align = False).values
    self.routes_summary.loc[subsample_idxs,'tort'] = self.routes_summary.loc[subsample_idxs,
    'length']/self.routes_summary.loc[subsample_idxs,'sl_dist']
    
    # self.routes_summary.loc[subsample_idxs,'bikeway_min_dist_orig'] = self.sample.loc[subsample_idxs].orig.shortest_line(self.n_ex_edges.union_all()).length
    # self.routes_summary.loc[subsample_idxs,'bikeway_min_dist_dest'] = self.sample.loc[subsample_idxs].dest.shortest_line(self.n_ex_edges.union_all()).length
    # all_grouped_route_links = self.all_routes_edges.groupby('route_number')
    # transit_min = all_grouped_route_links['transit_min_dist'].min()
    # self.routes_summary['transit_min_dist'] = self.routes_summary.index.map(transit_min)
    # bikeway_min = all_grouped_route_links['bikeway_min_dist'].min()
    # self.routes_summary['bikeway_min_dist'] = self.routes_summary.index.map(bikeway_min)
    
    # self.routes_summary.loc[subsample_idxs,'generalized_benefit']=( 
    # self.routes_summary.loc[subsample_idxs]['normalized_fpkm']+beta_age*(self.routes_summary.loc[subsample_idxs]['age_multiplier'])
    # +(self.routes_summary.loc[subsample_idxs]['transit_min_dist']<proximity_dist)*beta_transit
    # +(self.routes_summary.loc[subsample_idxs]['bikeway_min_dist']<proximity_dist)*beta_bikeway)

    self.routes_summary.loc[subsample_idxs,'generalized_benefit']=self.routes_summary.loc[subsample_idxs]['n_near_completed_routes']/self.routes_summary.loc[subsample_idxs,'length_street']
    if weight is not None:
        self.routes_summary.loc[subsample_idxs,'generalized_benefit']*= self.routes_summary.loc[subsample_idxs,weight] 
    
    # self.routes_summary.loc[subsample_idxs,'generalized_benefit']+=((self.routes_summary.loc[subsample_idxs]['transit_min_dist']<proximity_dist)*beta_transit*np.std(self.routes_summary.loc[subsample_idxs,'generalized_benefit'])
    #                                                                 +(self.routes_summary.loc[subsample_idxs]['bikeway_min_dist']<proximity_dist)*beta_bikeway*np.std(self.routes_summary.loc[subsample_idxs,'generalized_benefit']))
                                                                  
def compute_routes_replace(self,network,idxs,weight = 'gencost'):
    subsample = self.sample.loc[idxs]
    o_nodes,o_dists = ox.nearest_nodes(network,subsample.orig.x.values,subsample.orig.y.values, return_dist=True)
    d_nodes,d_dists = ox.nearest_nodes(network,subsample.dest.x.values,subsample.dest.y.values, return_dist=True)
    routes = ox.shortest_path(network, o_nodes, d_nodes, weight=weight)
    # for source, target in zip(o_nodes, d_nodes):
    #     route = shortest_path_turn_penalty.shortest_path_turn_penalty(network, source, target, weight, self.turn_penalties)
    #     routes.append(route)
    old_routes = self.routes.loc[idxs]
    self.routes.loc[idxs,'nodes']=pd.Series(data = routes,index=idxs)
    
    print(f"{1-(self.routes.loc[idxs].dropna(subset='nodes') == old_routes.dropna(subset='nodes')).mean().values[0]} % of routes changed, replacing...")
    
    route_edges = []
    for i in tqdm(range(len(idxs)),leave = False):
            if routes[i] is not None:
                if len(routes[i])>1:
                    edges = ox.routing.route_to_gdf(network,routes[i])
                    edges['route_number'] = idxs[i]
                    route_edges.append(edges)
    newroutes = pd.concat(route_edges)
    self.all_routes_edges = self.all_routes_edges[~self.all_routes_edges['route_number'].isin(idxs)]
    self.all_routes_edges = pd.concat([self.all_routes_edges,newroutes])
    
def reset_network(self):
    self.n_ex = self.n_ex_og.copy()
    self.n_ex_nodes,self.n_ex_edges = ox.graph_to_gdfs(self.n_ex)
    self.n_pot = self.n_pot_og.copy()
    self.n_pot_nodes,self.n_pot_edges = ox.graph_to_gdfs(self.n_pot)
    

def simulate_demand(self, size, method = 'uniform', max_dist = 5000):
    self.boundaries = gpd.GeoDataFrame({'name':['region']},geometry = gpd.GeoSeries(self.n_pot_edges.geometry.union_all()).concave_hull(0.2)
                                           , crs = self.crs)
    points = gpd.GeoDataFrame(self.boundaries.sample_points(size = size,method = method))
    points = points.set_geometry('sampled_points')
    points = points.explode()
    dests = []
    for point in points.sampled_points:
        circle = gpd.GeoDataFrame({'geometry':[point.buffer(max_dist)]}).set_crs(self.crs)
        dest_poly = gpd.overlay(circle,self.boundaries).geometry
        dest = dest_poly.sample_points(size=1).geometry.iloc[0]
        dests.append(dest)
    dests_gdf = gpd.GeoDataFrame({'geometry':dests}).set_crs(self.crs)

    self.sample = points
    self.sample = pd.concat([self.sample.reset_index(drop = True),dests_gdf.reset_index(drop = True)], axis = 1)
    self.sample = self.sample.rename(columns = {'sampled_points':'orig','geometry':'dest'})

def run_algo2(self, budget, savepath,beta_age, mu_age,scale_age,beta_transit,beta_bikeway,alpha,ltp,rtp,icp,tsp,build_dmin,weight = None,
              plot = False, buffer = 350, cost_reduction_factor=0.9, proximity_dist = 10, backup_every = 100):
    added_routes = []
    i=0
    if np.max(self.n_pot_edges.build_iter) == 0:
        # shutil.rmtree(savepath, ignore_errors=True)
        os.makedirs(savepath,exist_ok=True)
        params = {
        "budget": budget,
        "beta_age": beta_age,
        "mu_age": mu_age,
        "scale_age": scale_age,
        "beta_transit": beta_transit,
        "beta_bikeway": beta_bikeway,
        "cycleway_reduction_factor": cost_reduction_factor,
        "proximity_dist": proximity_dist,
        "buffer": buffer,
        "rtp": ltp,
        "ltp": rtp,
        "icp": icp,
        "tsp":tsp,
        "alpha":alpha}
        df_params = pd.DataFrame(list(params.items()), columns=["Parameter", "Value"])
        df_params.to_csv(savepath + "/params.csv", index=False)
        
        print('Initializing, computing desire boxes...')
        if 'bbox' not in self.sample.columns:
            self.od_bbox(buffer = buffer)
        self.n_pot_edges['gencost'] = self.n_pot_edges['length']
        self.weight_network(cycleway_reduc_factor=cost_reduction_factor, grades=True)
        # self.get_personal_spatial_features(mu_age,scale_age)
        #self.get_turn_penalties(self.n_pot,icp = icp,ltp = ltp,rtp = rtp,tsp = tsp)
        self.reset_routes()
        self.evol = []
        print('Computing routes...')
        self.compute_routes(self.n_pot)
        self.get_routes_edges(self.n_pot)
        print('Computing route metrics...')
        self.compute_routes_summary(beta_age = beta_age,
                                    mu_age = mu_age,
                                    scale_age = scale_age,
                                    beta_transit = beta_transit,
                                    beta_bikeway = beta_bikeway,
                                    alpha = alpha,
                                    proximity_dist = proximity_dist,
                                    weight = weight)
        self.evol.append(self.routes_summary.copy())
    else:
        i=np.max(self.n_pot_edges.build_iter)
    while self.n_ex_edges[self.n_ex_edges.build_iter > 0].length.sum()/1000 < budget:
        
        if (i%backup_every==0)&(i!=0):
            self.save_network(savepath = savepath,i=i,show = False)
            self.routes_summary.to_csv(savepath + f"/summary_{i}.csv", index=False)
        i+=1
        if plot:
            if i==0:
                fig, ax = plt.subplots(figsize = (5,5))
                ax.axis('off')
        clear_output(wait = True)
        print(f'Built {self.n_ex_edges[self.n_ex_edges.build_iter > 0].length.sum()/1000}/{str(budget)} km (iteration {i})')

        route_id_to_add = self.routes_summary[(self.routes_summary.length_street>build_dmin)&(self.routes_summary.length_cycleway>0)]['generalized_benefit'].idxmax()
        print('Adding route ', route_id_to_add, f'(benefit = {self.routes_summary.loc[route_id_to_add,"generalized_benefit"]},length = {self.routes_summary.loc[route_id_to_add,"length_street"]})')
        route_edges = self.all_routes_edges[self.all_routes_edges.route_number == int(route_id_to_add)]
        route_edges_to_add = route_edges[route_edges.highway!='cycleway']
        edges_id_to_add = route_edges_to_add.index
        self.n_pot_edges.loc[edges_id_to_add,'highway']='cycleway'
        self.n_pot_edges.loc[edges_id_to_add,'build_iter'] = np.max(self.n_pot_edges.build_iter)+1
        self.n_ex_edges = self.n_pot_edges[self.n_pot_edges.highway == 'cycleway']
        self.n_pot_edges['bikeway_min_dist'] = self.n_pot_edges.shortest_line(self.n_ex_edges.union_all()).length
        self.all_routes_edges.loc[edges_id_to_add,'highway']='cycleway'

        self.weight_network(cost_reduction_factor, grades = True)
        routes_int_idxs = self.associated_links.loc[edges_id_to_add].values
        routes_int_idxs = list(set(routes_int_idxs) & set(self.sample.index.values))
   
        self.evol.append(self.routes_summary.copy())
        
        if len(routes_int_idxs) == 0:
            self.compute_routes_replace(self.n_pot,[route_id_to_add])
        else:
            print('recomputing ',len(routes_int_idxs)/len(self.sample),f'({len(routes_int_idxs)}/{len(self.sample)})')
            self.compute_routes_replace(self.n_pot,list(set(routes_int_idxs)-set(added_routes)))
        print('Computing route metrics...')
        added_routes.append(route_id_to_add)
        self.compute_routes_summary(beta_age = beta_age,
                                    mu_age = mu_age,
                                    scale_age = scale_age,
                                    beta_transit = beta_transit,
                                    beta_bikeway = beta_bikeway,
                                    proximity_dist = proximity_dist,
                                    alpha = alpha,
                                    weight = weight,
                                    subsample_idxs = list(set(routes_int_idxs)|set([route_id_to_add])))
    self.save_network(savepath = savepath,i='final')
    self.routes_summary.to_csv(savepath + "/summary.csv", index=False)

def add_edge_bearing(self):
    n_pot_unproj = ox.projection.project_graph(self.n_pot,to_crs=4326)
    n_pot_angle = ox.bearing.add_edge_bearings(n_pot_unproj)
    nodes,edges = ox.graph_to_gdfs(n_pot_angle)
    self.n_pot_edges['bearing'] = edges['bearing'].fillna(0)
    self.n_pot = ox.graph_from_gdfs(self.n_pot_nodes,self.n_pot_edges)
    self.n_pot_og = ox.graph_from_gdfs(self.n_pot_nodes.copy(),self.n_pot_edges.copy())
def od_bbox(self,buffer):
    def compute_bounding_box(row):
        min_x = min(row["orig"].x, row["dest"].x)
        max_x = max(row["orig"].x, row["dest"].x)
        min_y = min(row["orig"].y, row["dest"].y)
        max_y = max(row["orig"].y, row["dest"].y)
        return box(min_x, min_y, max_x, max_y).buffer(buffer)
        
    
    def get_links_bbox(row):
        links = gpd.sjoin(row,self.n_pot_edges,how = 'left',
                          predicate = 'intersects').set_index(['u', 'v', 'key'])['ipere']

    n_pot_unproj = ox.projection.project_graph(self.n_pot,to_crs=4326)
    n_pot_angle = ox.bearing.add_edge_bearings(ox.convert.to_undirected(n_pot_unproj))
    nodes,edges = ox.graph_to_gdfs(n_pot_angle)
    edges = edges.dropna(subset=['bearing'])
    # lengths = pd.merge(edges,self.n_pot_edges,how = 'inner')['length']
    dom_angle = np.average(edges.bearing.values%90,weights = edges['length'])
    print('Dominating angle:', dom_angle)
    # dom_angle = 0
    centroid = self.sample.union_all().centroid
    rotated_sample_o = self.sample.orig.rotate(dom_angle,centroid).explode()
    rotated_sample_d = self.sample.dest.rotate(dom_angle,centroid).explode()
    rotated_sample_o.name = 'orig'
    rotated_sample_d.name = 'dest'
    rotated_sample = pd.concat([rotated_sample_o,rotated_sample_d],axis = 1)
    rotated_sample = gpd.GeoDataFrame(rotated_sample,geometry='orig')
    rotated_sample["bbox"] = rotated_sample.apply(compute_bounding_box, axis=1)
    rotated_sample.set_crs(self.crs)
    self.sample['bbox'] = rotated_sample.bbox.rotate(-dom_angle,centroid)
    self.sample = self.sample.set_geometry('bbox').set_crs(self.crs)
    self.associated_links = gpd.sjoin(self.sample,self.n_pot_edges,how = 'left',
                          predicate = 'intersects').set_index(['u', 'v', 'key'])['ipere']

def weight_network(self,cycleway_reduc_factor=0.9, grades = False):
    self.n_pot_edges['gencost'] = self.n_pot_edges['length']*self.n_pot_edges["highway"].apply(
        lambda x: cycleway_reduc_factor if x == "cycleway" else 1)
    if grades:
        self.n_pot_edges.loc[self.n_pot_edges.length>15,'gencost'] *= (1*(self.n_pot_edges.grade<0.02)+1.371*((self.n_pot_edges.grade>=0.02)&(self.n_pot_edges.grade<0.04))+
                                  2.203*((self.n_pot_edges.grade>=0.04)&(self.n_pot_edges.grade<0.06))+
                                  4.239*((self.n_pot_edges.grade>=0.06)))
                                 
    self.n_pot = ox.graph_from_gdfs(self.n_pot_nodes,self.n_pot_edges)

# def weight_network(self, cycleway_reduc_factor=0.9, grades=False):
#     for u, v, k, data in self.n_pot.edges(keys=True, data=True):
#         # base cost
#         cost = data["length"] * (cycleway_reduc_factor if data.get("highway") == "cycleway" else 1)

#         if grades and data["length"] > 15:
#             g = data.get("grade", 0)
#             if g < 0.02:
#                 factor = 1
#             elif g < 0.04:
#                 factor = 1.371
#             elif g < 0.06:
#                 factor = 2.203
#             else:
#                 factor = 4.239
#             cost *= factor

#         data["gencost"] = cost


def add_edge_grades(self,elevation_filepath):
    self.n_pot = ox.elevation.add_node_elevations_raster(self.n_pot,elevation_filepath)
    self.n_pot = ox.elevation.add_edge_grades(self.n_pot,add_absolute=True)
    self.n_ex = ox.elevation.add_node_elevations_raster(self.n_ex,elevation_filepath)
    self.n_ex = ox.elevation.add_edge_grades(self.n_ex,add_absolute=True)
    self.n_pot_nodes,self.n_pot_edges = ox.graph_to_gdfs(self.n_pot)
    self.n_ex_nodes,self.n_ex_edges = ox.graph_to_gdfs(self.n_ex)
    self.n_pot_edges['build_iter'] = 0
    self.n_ex_edges['build_iter'] = 0
    self.n_pot_og = ox.graph_from_gdfs(self.n_pot_nodes.copy(),self.n_pot_edges.copy())
    self.n_ex_og = ox.graph_from_gdfs(self.n_ex_nodes.copy(),self.n_ex_edges.copy())

def display_network(self, tiles = 'OpenStreetMap',savepath = None, max_iter = None,color = 'red',show = True):
    built_edges = self.n_ex_edges[self.n_ex_edges.build_iter > 0]
    existing_edges = self.n_ex_edges[self.n_ex_edges.build_iter == 0]
    if max_iter is not None:
        built_edges = built_edges[built_edges.build_iter<=max_iter]
          
    new_network = built_edges.explore(tiles = tiles,
        column='build_iter',
        cmap='viridis_r',
        style_kwds={'weight': 4},
        legend=True
    )
    
    existing_edges.explore(
        color=color,
        style_kwds={'weight': 2, 'opacity': 0.5},
        m=new_network
    )
    
    dummy = existing_edges.head(1).copy()
    dummy['Legend'] = 'Existing network'
    dummy.explore(
        column='Legend',
        cmap=[color],
        m=new_network,
        legend=True
    )

    Fullscreen(position='topright').add_to(new_network)
    smooth_zoom_js = """
    <script>
        var map = {{this._parent.get_name()}};
        map.options.zoomSnap = 0.1;
        map.options.zoomDelta = 0.1;
    </script>
    """
    
    new_network.get_root().html.add_child(folium.Element(smooth_zoom_js))
    if show:
        display(new_network)
        print(f'length = {built_edges.length.sum()/1000} km')
    if savepath is not None:
        new_network.save(savepath)

    return new_network

def compute_routes(self, network, weight = 'gencost'):
    network = ox.projection.project_graph(network, to_crs = self.crs)
    o_nodes,o_dists = ox.nearest_nodes(network,self.sample.orig.x.values,self.sample.orig.y.values, return_dist=True)
    d_nodes,d_dists = ox.nearest_nodes(network,self.sample.dest.x.values,self.sample.dest.y.values, return_dist=True)
    routes = ox.shortest_path(network, o_nodes, d_nodes, weight=weight,cpus = None)
    self.routes = pd.DataFrame(data = {'nodes': routes}, index = self.sample.index)
    
def compute_routes_turn_penalty(self, network, weight = 'gencost'):
    network = ox.projection.project_graph(network, to_crs = self.crs)
    o_nodes,o_dists = ox.nearest_nodes(network,self.sample.orig.x.values,self.sample.orig.y.values, return_dist=True)
    d_nodes,d_dists = ox.nearest_nodes(network,self.sample.dest.x.values,self.sample.dest.y.values, return_dist=True)
    routes = []
    for source, target in zip(o_nodes, d_nodes):
        route = shortest_path_turn_penalty.shortest_path_turn_penalty(network, source, target, weight, self.turn_penalties)
        routes.append(route)
    self.routes = pd.DataFrame(data = {'nodes': routes}, index = self.sample.index)
    
def save_network(self,savepath,i,show = True):
    print('Network backup at '+ savepath)
    ox.io.save_graphml(self.n_pot,filepath = savepath+f'/network_it_{i}.graphml')
    self.display_network(savepath = savepath+f'/network_it_{i}.html',show = show)

def get_personal_spatial_features(self,mu_age,scale_age):
    self.n_pot_edges['bikeway_min_dist'] = self.n_pot_edges.shortest_line(self.n_ex_edges.union_all()).length
    self.n_pot_edges['transit_min_dist'] = self.n_pot_edges.shortest_line(self.transit_stops.union_all()).length
    self.n_pot = ox.graph_from_gdfs(self.n_pot_nodes,self.n_pot_edges)
    age_multiplier = scipy.stats.gumbel_l(loc = mu_age,scale = scale_age).pdf(self.sample['age'])
    age_multiplier/=np.max(age_multiplier)
    age_multiplier = 2-age_multiplier
    self.sample['age_multiplier'] = age_multiplier

def get_turn_penalties(self,network,icp,ltp,rtp,tsp):
    self.turn_penalties = penalty_turns(network,left_turn_penalty=ltp,right_turn_penalty=rtp,intersection_crossing_penalty=icp,traffic_signals_penalty=tsp)
    
funcToMethod(run_algo,Network,method_name = 'run_algo')
funcToMethod(compute_routes_summary,Network,method_name = 'compute_routes_summary')
funcToMethod(compute_routes_replace,Network,method_name = 'compute_routes_replace')
funcToMethod(reset_network,Network,method_name = 'reset_network')
funcToMethod(simulate_demand,Network,method_name = 'simulate_demand')
funcToMethod(run_algo2,Network,method_name = 'run_algo2')
funcToMethod(od_bbox,Network,method_name = 'od_bbox')
funcToMethod(weight_network,Network,method_name = 'weight_network')
funcToMethod(add_edge_grades,Network,method_name = 'add_edge_grades')
funcToMethod(display_network,Network,method_name = 'display_network')
funcToMethod(compute_routes,Network,method_name = 'compute_routes')
funcToMethod(save_network,Network,method_name = 'save_network')
funcToMethod(get_personal_spatial_features,Network,method_name = 'get_personal_spatial_features')
funcToMethod(add_edge_bearing,Network,method_name = 'add_edge_bearing')
funcToMethod(compute_routes_turn_penalty,Network,method_name = 'compute_routes_turn_penalty')
funcToMethod(get_turn_penalties,Network,method_name = 'get_turn_penalties')


In [ ]:
cost_reduction_factor = 0.7
np.random.seed(123)
n.reset_network()
n.add_edge_grades('Data/Elevation/output_AW3D30_32618_us.tif')
n.reset_routes()
n.reset_sample()
# n.trips_within = n.trips_within[n.trips_within['mode']!='TC']
n.weight_network(cycleway_reduc_factor=cost_reduction_factor,grades=True)
n.sample = n.trips_within.sort_values('ntrips',ascending = False).iloc[:5000]
n.sample.index = n.sample.ipere.values

In [6]:
signals = pd.read_csv(signals_file)
signals = gpd.GeoDataFrame(signals,geometry = gpd.points_from_xy(signals.Longitude,signals.Latitude),crs = 4326).to_crs(n.crs)
n.traffic_signals=signals
near_traffic_signals = gpd.sjoin_nearest(n.n_pot_nodes,n.traffic_signals,max_distance=15,how = 'left')
near_traffic_signals = near_traffic_signals[~near_traffic_signals.index.duplicated(keep='first')]
n.n_pot_nodes['traffic_signals'] = near_traffic_signals.index_right.notna()

ValueError: 'left_df' should be GeoDataFrame, got <class 'NoneType'>

In [ ]:
# n.get_personal_spatial_features(1,1)
n.add_edge_bearing()
n.reset_routes()

In [ ]:
plt.figure()
sns.histplot(n.routes_summary[(n.routes_summary.potVelo==1)|(n.routes_summary.velo=='X')],
             x = 'prop_cycleway',hue = 'velo',common_norm = False,stat = 'proportion',fill = False,)
plt.legend(title = 'Demande',labels = ['Potentielle','Observée'], fontsize = 14)

plt.xlabel('Proportion protégée',fontsize = 14)
plt.ylabel('Proportion',fontsize = 14)

plt.tight_layout()
plt.savefig('Figures/prop_cycleway_latent_observed.png')

In [ ]:
plt.figure()
sns.histplot(n.routes_summary[(n.routes_summary.potVelo==1)|(n.routes_summary.velo=='X')],
             x = 'length',hue = 'velo',common_norm = False,stat = 'proportion',fill = True,binwidth = 1000)
plt.legend(title = 'Demande',labels = ['Potentielle','Observée'], fontsize = 14)

plt.xlabel('Longueur du trajet (m)',fontsize = 14)
plt.ylabel('Proportion',fontsize = 14)

plt.tight_layout()
plt.savefig('Figures/length_latent_observed.png')

In [ ]:
bins = [0, 18, 40, 100]  # you can adjust
labels = ["0-17", "18-39", "40+"]

n.routes_summary["age group"] = pd.cut(n.routes_summary["age"], bins=bins, labels=labels, right=True)
plt.figure()
sns.histplot(n.routes_summary,x = 'prop_cycleway',hue = 'age group',fill = False,stat = 'proportion')
plt.semilogy()

plt.xlabel('On cycleway proportion',fontsize = 18)
plt.ylabel('Proportion of trips',fontsize = 18)
plt.tight_layout()
plt.savefig('../Conferences/ATC2025/Figures/profrac_age.png')

In [ ]:
def indicators_summary(summary):
    tortbins = np.linspace(1,3,11)
    
    fig, ax = plt.subplots(3,2,figsize = (10,5))
    sns.ecdfplot(summary,x = 'tort',ax = ax[0][0],stat = 'proportion',log_scale=True)
    sns.histplot(summary,x = 'prop_cycleway',ax = ax[0][1],stat = 'proportion')
    turns_bins = np.histogram_bin_edges(pd.concat([summary.n_left_turns_per_km,summary.n_right_turns_per_km]),10)
    sns.histplot(summary,x = 'n_left_turns_per_km',ax = ax[1][0],bins = turns_bins,label = 'Gauche',stat = 'proportion')
    sns.histplot(summary,x = 'n_right_turns_per_km',ax = ax[1][1],bins = turns_bins,label = 'Droite',stat = 'proportion')
    dist_bins = np.histogram_bin_edges(pd.concat([summary.bikeway_min_dist_orig,summary.bikeway_min_dist_dest]),12)
    sns.histplot(summary,x = 'bikeway_min_dist_orig',ax = ax[2][0],bins = dist_bins,label = 'Origine',stat = 'proportion')
    sns.histplot(summary,x = 'bikeway_min_dist_dest',ax = ax[2][1],bins = dist_bins,label = 'Destination',stat = 'proportion')
    ax[0][0].set_xlabel('Tortuosité')
    ax[0][0].set_ylabel('Proportion\ncumulée')
    ax[0][1].set_xlabel('Proportion protégée')
    ax[1][0].set_xlabel('Nombre de virages à gauche par km')
    ax[1][1].set_xlabel('Nombre de virages à droite par km')
    ax[2][1].set_xlabel('Distance entre la destination\net le réseau cyclable (m)')
    ax[2][0].set_xlabel('Distance entre l\'origine\net le réseau cyclable (m)')
    
    fig.tight_layout()
    return fig

fig = indicators_summary(n.routes_summary)
fig.savefig('Figures/sit_ini_indicateurs.png')

In [ ]:
def compute_accessibility(network,dests,domis, dist, buffer):
    nodes,edges = ox.graph_to_gdfs(network)
    source_nodes = ox.nearest_nodes(network,dests.geometry.x.values,dests.geometry.y.values,return_dist=False)
    edges['weight_bike']= edges.apply(lambda x: np.inf if x['highway'] !='cycleway' else x['gencost'], axis = 1)
    edges['weight_bike_init']= edges.apply(lambda x: np.inf if x['build_iter'] != 0 else x['weight_bike'], axis = 1)
    network = ox.graph_from_gdfs(nodes,edges)
    dests['access_potential_ratio']=0
    dests['access_increase_ratio']=0
    for i,source_node in enumerate(source_nodes):
        clear_output(wait = True)
        print(f'{i+1}/{len(source_nodes)}')
        n_domis = []
        for weight in ['weight_bike','length','weight_bike_init']:
            paths = nx.single_source_dijkstra(network,source = source_node, cutoff = dist, weight = weight)
            reachable_nodes = paths[0].keys()
            reachable_edges = edges.sjoin(nodes.loc[reachable_nodes],how = 'inner',predicate = 'intersects')
            counts = reachable_edges.groupby(reachable_edges.index).size()
            reachable_edges = edges.loc[counts[counts >= 2].index]
            catchment_area = gpd.GeoDataFrame({'name':['catchment_area']},geometry = gpd.GeoSeries(reachable_edges.geometry.union_all()).buffer(buffer)
                                                   , crs = network.graph['crs'])
            domis_access = domis.sjoin(catchment_area,how = 'inner',predicate = 'intersects')
            n_domis.append(domis_access.faclog_s27.sum())
        dests.iloc[i].access_potential_ratio = n_domis[0]/n_domis[1]
        if n_domis[2]!=0:
            dests.iloc[i].access_increase_ratio= n_domis[0]/n_domis[2]
        else:
            if n_domis[0]!=0:
                dests.iloc[i].access_increase_ratio = np.inf
            else:
                dests.iloc[i].access_increase_ratio = 1
    return dests

In [ ]:
signals = pd.read_csv(signals_file)

def add_traffic_signals(signals,network):
    signals = gpd.GeoDataFrame(signals,geometry = gpd.points_from_xy(signals.Longitude,signals.Latitude),crs = 4326).to_crs(n.crs)
    network.traffic_signals=signals
    near_traffic_signals = gpd.sjoin_nearest(n.n_pot_nodes,n.traffic_signals,max_distance=15,how = 'left')
    near_traffic_signals = near_traffic_signals[~near_traffic_signals.index.duplicated(keep='first')]
    network.n_pot_nodes['traffic_signals'] = near_traffic_signals.index_right.notna()

add_traffic_signals(signals,n)

In [ ]:
cost_reduction_factor = 0.7
proximity_dist = 10
beta_age = 0
beta_bikeway = 5
beta_transit = 1
icp = 50
ltp = 6*icp
rtp = 2*icp
tsp = 2*icp
mu_age = 25
scale_age = 5
alpha = .2
B = 30

name = input('Filename:')
date_time_str = datetime.now().strftime('%m-%d')
n.run_algo2(budget = B,savepath=f'Figures/reseaux/{name}-{date_time_str}',
            buffer = 500,icp = icp, ltp = ltp,rtp = rtp,tsp = tsp,
            cost_reduction_factor = cost_reduction_factor,proximity_dist=proximity_dist,beta_age = beta_age,mu_age = mu_age,scale_age = scale_age, 
            beta_bikeway = beta_bikeway,
            beta_transit = beta_transit,
            alpha = alpha,
            backup_every=10,weight = 'ntrips')

In [ ]:
n_mtl = Network()
ex = 'Data/Reseaux/ns_EX_MTL.graphml'
pot = 'Data/Reseaux/ns_POT_MTL.graphml'
trips = 'Data/od18_extraqit_20250123/od18_extraqit_20250123.csv'
stm_gtfs = 'Data/gtfs_stm/stops.txt'
signals_file = 'Data/feux-circulation.csv'

n_mtl.load_n_pot(pot)
n_mtl.load_n_ex(ex)
n_mtl.load_trips(trips,source_crs=2950)

stops = pd.read_csv(stm_gtfs)
metros = stops[stops['location_type']==1]
metros = gpd.GeoDataFrame(metros,geometry = gpd.points_from_xy(metros.stop_lon,metros.stop_lat),crs = 4326).to_crs(n_mtl.crs)
n_mtl.transit_stops=metros

n_mtl.reset_network()
n_mtl.add_edge_grades('Data/Elevation/output_AW3D30_32618_us.tif')
n_mtl.reset_routes()
n_mtl.reset_sample()
n_mtl.trips_within = n_mtl.trips_within[n_mtl.trips_within['mode']!='TC']
n_mtl.weight_network(cycleway_reduc_factor=cost_reduction_factor,grades=True)
n_mtl.sample_trips()
n_mtl.get_personal_spatial_features(1,1)
n_mtl.add_edge_bearing()

In [ ]:
cost_reduction_factor = 0.7
proximity_dist = 10
beta_age = 0
beta_bikeway = 5
beta_transit = 1
icp = 50
ltp = 6*icp
rtp = 2*icp
tsp = 2*icp
mu_age = 25
scale_age = 5
alpha = .2
B = 60

name = input('Filename:')
date_time_str = datetime.now().strftime('%m-%d')
n_mtl.run_algo2(budget = B,savepath=f'Figures/reseaux/{name}-{date_time_str}',
            buffer = 500,icp = icp, ltp = ltp,rtp = rtp,tsp = tsp,
            cost_reduction_factor = cost_reduction_factor,proximity_dist=proximity_dist,beta_age = beta_age,mu_age = mu_age,scale_age = scale_age, 
            beta_bikeway = beta_bikeway,
            beta_transit = beta_transit,
            alpha = alpha,
            backup_every=10)

In [44]:
n_lvl = Network()
ex = 'Data/Reseaux/ns_EX_Laval,Qc.graphml'
pot = 'Data/Reseaux/ns_POT_Laval,Qc.graphml'
trips = 'Data/od18_extraqit_20250123/od18_extraqit_20250123.csv'
stm_gtfs = 'Data/gtfs_stm/stops.txt'
signals_file = 'Data/feux-circulation.csv'

n_lvl.load_n_pot(pot)
n_lvl.load_n_ex(ex)
n_lvl.load_trips(trips,source_crs=2950)

stops = pd.read_csv(stm_gtfs)
metros = stops[stops['location_type']==1]
metros = gpd.GeoDataFrame(metros,geometry = gpd.points_from_xy(metros.stop_lon,metros.stop_lat),crs = 4326).to_crs(n_lvl.crs)
n_lvl.transit_stops=metros

n_lvl.reset_network()
n_lvl.add_edge_grades('Data/Elevation/output_AW3D30_32618_us.tif')
n_lvl.reset_routes()
n_lvl.reset_sample()
n_lvl.trips_within = n_lvl.trips_within[n_lvl.trips_within['mode']!='TC']
n_lvl.weight_network(cycleway_reduc_factor=cost_reduction_factor,grades=True)
n_lvl.sample_trips()
n_lvl.get_personal_spatial_features(1,1)
n_lvl.add_edge_bearing()

NameError: name 'cost_reduction_factor' is not defined

In [ ]:
name = input('Filename:')
date_time_str = datetime.now().strftime('%m-%d')
n_lvl.run_algo2(budget = B,savepath=f'Figures/reseaux/{name}-{date_time_str}',
            buffer = 500,icp = icp, ltp = ltp,rtp = rtp,tsp = tsp,
            cost_reduction_factor = cost_reduction_factor,proximity_dist=proximity_dist,beta_age = beta_age,mu_age = mu_age,scale_age = scale_age, 
            beta_bikeway = beta_bikeway,
            beta_transit = beta_transit,
            alpha = alpha,
            backup_every=10)

In [ ]:
n_qc = Network(crs = 32619)
ex = 'Data/Reseaux/ns_EX_QC.graphml'
pot = 'Data/Reseaux/ns_POT_QC.graphml'
trips = 'Data/od17QC_extrait_20250522/od17QC_extrait_20250522.csv'
stm_gtfs = 'Data/gtfs_stm/stops.txt'
signals_file = 'Data/feux-circulation.csv'

n_qc.load_n_pot(pot)
n_qc.load_n_ex(ex)
n_qc.load_trips(trips,source_crs=2949,delimiter=';',cols=['ipere', 'mode', 'motif', 'age', 'xorig', 'yorig', 'xdest', 'ydest', 'potVelo', 'fexpPotVelo', 'velo'])

stops = pd.read_csv(stm_gtfs)
metros = stops[stops['location_type']==1]
metros = gpd.GeoDataFrame(metros,geometry = gpd.points_from_xy(metros.stop_lon,metros.stop_lat),crs = 4326).to_crs(n_qc.crs)
n_qc.transit_stops=metros


In [ ]:
n_qc.reset_network()
n_qc.add_edge_grades('Data/Elevation/output_USGS30m_32619.tif')
n_qc.reset_routes()
n_qc.reset_sample()
n_qc.trips_within = n_qc.trips_within[n_qc.trips_within['mode']!='TC']
n_qc.weight_network(cycleway_reduc_factor=cost_reduction_factor,grades=True)
n_qc.sample_trips()
n_qc.get_personal_spatial_features(1,1)
n_qc.add_edge_bearing()

In [ ]:
B=60
name = input('Filename:')
date_time_str = datetime.now().strftime('%m-%d')
n_qc.run_algo2(budget = B,savepath=f'Figures/reseaux/{name}-{date_time_str}',
            buffer = 500,icp = icp, ltp = ltp,rtp = rtp,tsp = tsp,
            cost_reduction_factor = cost_reduction_factor,proximity_dist=proximity_dist,beta_age = beta_age,mu_age = mu_age,scale_age = scale_age, 
            beta_bikeway = beta_bikeway,
            beta_transit = beta_transit,
            alpha = alpha,
            build_dmin = 0,
            backup_every=10)

In [ ]:
n_qc.display_network()

In [ ]:
plt.rcParams.update({'font.size': 13})

plt.figure()

lengths_mtl = [n_mtl.n_ex_edges[n_mtl.n_ex_edges.build_iter<i].length.sum() for i in range(1,n_mtl.n_ex_edges.build_iter.max()+1)]
lengths_lvl = [n_lvl.n_ex_edges[n_lvl.n_ex_edges.build_iter<i].length.sum() for i in range(1,n_lvl.n_ex_edges.build_iter.max()+1)]
lengths_qc = [n_qc.n_ex_edges[n_qc.n_ex_edges.build_iter<i].length.sum() for i in range(1,n_qc.n_ex_edges.build_iter.max()+1)]

plt.plot((np.array(lengths_mtl)-lengths_mtl[0])/1000,label = 'Laval')
plt.plot((np.array(lengths_lvl)-lengths_lvl[0])/1000,label = 'Montreal')
plt.plot((np.array(lengths_qc)-lengths_qc[0])/1000,label = 'Quebec')

plt.legend()
plt.xlabel('Iteration')
plt.ylabel('Built length (km)')

plt.savefig('Figures/lvl_mtl_qc_length_iteration.png')

In [ ]:
fig,ax = plt.subplots(1,2,figsize = (10,4))

cr_mtl = [(e.prop_cycleway>0.99).sum() for e in n_mtl.evol]
cr_lvl = [(e.prop_cycleway>0.99).sum() for e in n_lvl.evol]
cr_qc = [(e.prop_cycleway>0.99).sum() for e in n_qc.evol]


ax[0].plot((lengths_lvl-lengths_lvl[0])/1000,cr_lvl[1:])
ax[0].plot((lengths_mtl-lengths_mtl[0])/1000,cr_mtl[1:-1])
ax[0].plot((lengths_qc-lengths_qc[0])/1000,cr_qc[1:])


ax[1].plot((lengths_lvl-lengths_lvl[0])/1000,np.array(cr_lvl[1:])/len(n_lvl.evol[0]),label = 'Laval')
ax[1].plot((lengths_mtl-lengths_mtl[0])/1000,np.array(cr_mtl[1:-1])/len(n_mtl.evol[0]),label = 'Montreal')
ax[1].plot((lengths_qc-lengths_qc[0])/1000,np.array(cr_qc[1:])/len(n_qc.evol[0]),label = 'Quebec')


ax[0].set_xlabel('Built length (km)')
ax[1].set_xlabel('Built length (km)')

ax[0].set_ylabel('Number of completed routes')
ax[1].set_ylabel('Relative number of\ncompleted routes')
ax[1].legend()

fig.tight_layout()
fig.savefig('Figures/lvl_mtl_qc_cr_length.png')

In [ ]:
fig, ax = plt.subplots(1,3,figsize = (13,4))
bins = 12
binrange = (0,1)
sns.histplot(n_mtl.evol[0], x = 'prop_cycleway',fill = False,ax = ax[0],bins = bins,binrange = binrange,label = 'Initial')
sns.histplot(n_mtl.evol[-1], x = 'prop_cycleway',fill = False,ax = ax[0],bins = bins,binrange = binrange,label = 'Final',color = 'red')

sns.histplot(n_lvl.evol[0], x = 'prop_cycleway',fill = False,ax = ax[1],bins = bins,binrange = binrange,label = 'Initial')
sns.histplot(n_lvl.evol[-1], x = 'prop_cycleway',fill = False,ax = ax[1],bins = bins,binrange = binrange,label = 'Final',color = 'red')

sns.histplot(n_qc.evol[0], x = 'prop_cycleway',fill = False,ax = ax[2],bins = bins,binrange = binrange,label = 'Initial')
sns.histplot(n_qc.evol[-1], x = 'prop_cycleway',fill = False,ax = ax[2],bins = bins,binrange = binrange,label = 'Final',color = 'red')

ax[0].set_xlabel('On cycleway proportion')
ax[1].set_xlabel('On cycleway proportion')
ax[2].set_xlabel('On cycleway proportion')
ax[1].set_ylabel(None)
ax[2].set_ylabel(None)


ax[0].set_title('Montreal')
ax[1].set_title('Laval')
ax[2].set_title('Quebec')


# ax[0].legend()
ax[2].legend()

fig.tight_layout()
fig.savefig('Figures/dist_cr_lvl_mtl_qc.png')

In [ ]:
cost_reduction_factor = 0.7
n.reset_network()
n.add_edge_grades('Data/Elevation/output_AW3D30_32618_us.tif')
n.reset_routes()
n.reset_sample()
n.trips_within = n.trips_within[n.trips_within['mode']!='TC']
n.weight_network(cycleway_reduc_factor=cost_reduction_factor,grades=True)
n.sample_trips()
n.get_personal_spatial_features(1,1)
n.add_edge_bearing()


In [ ]:
B = 50
buffer = 500
icps = [20,5,10,15,25,30,35,40,45,50]
crfs = [0.7,0.1,0.2,0.3,0.4,0.5,0.6,0.8,0.9]
proximity_dists = [10,30,50,70,90,120]
beta_bikeways = [5,0,1,2,3,4,6,7,8,9,10]
beta_transits = [1,0,2,3,4,5,6,7,8,9,10]
alphas = [0.2,0,0.05,0.1,0.15,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7]
date_time_str = datetime.now().strftime('%m-%d')
v_params = [icps,crfs,proximity_dists,beta_bikeways,beta_transits,alphas]
m=0
for i,v_p in enumerate(v_params):
    for j in range(len(v_p)): 
        if j == 0:
            m+=1
            if i != 0:
                continue
        params = [p[0] for p in v_params[:m-1]]+[v_params[m-1][j]]+[p[0] for p in v_params[m:]]
        params_str = '-'.join(list(map(str, params)))
        icp = params[0]
        ltp = 6*icp
        rtp = 2*icp
        tsp = 2*icp

        n.reset_network()
        add_traffic_signals(signals,n)
        n.run_algo2(budget = B,savepath=f'Figures/reseaux/sens_test-{date_time_str}/{params_str}',
            buffer = buffer,icp = icp, ltp = ltp,rtp = rtp,tsp = tsp,
            cost_reduction_factor = params[1],proximity_dist=params[2],beta_age = 0,mu_age = 25,scale_age = 5, 
            beta_bikeway = params[3],
            beta_transit = params[4],
            alpha = params[5],
            backup_every=10)

In [ ]:
n_qc = Network(crs = 32619)
ex = 'Data/Reseaux/ns_EX_QC.graphml'
pot = 'Data/Reseaux/ns_POT_QC.graphml'
trips = 'Data/od17QC_extrait_20250522/od17QC_extrait_20250522.csv'
stm_gtfs = 'Data/gtfs_stm/stops.txt'
n_qc.load_n_pot(pot)
n_qc.load_n_ex(ex)
n_qc.load_trips(trips,source_crs=2949,delimiter=';',cols=['ipere', 'mode', 'motif', 'age', 'xorig', 'yorig', 'xdest', 'ydest', 'potVelo', 'fexpPotVelo', 'velo'])
stops = pd.read_csv(stm_gtfs)
metros = stops[stops['location_type']==1]
metros = gpd.GeoDataFrame(metros,geometry = gpd.points_from_xy(metros.stop_lon,metros.stop_lat),crs = 4326).to_crs(n.crs)
n_qc.transit_stops=metros


In [7]:
input_tif = 'Data/Elevation/output_USGS30m.tif'
output_tif = 'Data/Elevation/output_USGS30m_32619.tif'
dst_crs = 32619
convert_tif_crs(input_tif,output_tif,dst_crs)

In [114]:
n_lev = Network(crs=32619)
ex = 'Data/Reseaux/ns_EX_Lévis.graphml'
pot = 'Data/Reseaux/ns_POT_Lévis.graphml'
trips = 'Data/od23QC_extrait_20251002/od23QC_extrait_20251002.csv'
stm_gtfs = 'Data/gtfs_stm/stops.txt'
n_lev.load_n_pot(pot)
n_lev.load_n_ex(ex)
n_lev.load_trips(trips, source_crs=2949,potvel = False, cols=['ipere', 'mode', 'motif_2017', 'age', 'xorig', 'yorig', 'xdest', 'ydest', 'potVelo', 'fexpPotVelo', 'velo'])

ValueError: `gdf_edges` must be multi-indexed by `(u, v, key)`.

In [58]:
reseau_lev = gpd.read_file('Data/Ville Levis/piste-cyclable.json')

In [109]:
n_lev.transit_stops=metros
n_lev.reset_network()
n_lev.add_edge_grades('Data/Elevation/output_USGS30m_32619.tif')
n_lev.reset_routes()
n_lev.reset_sample()
n_lev.trips_within = n_lev.trips_within[n_lev.trips_within['mode']!='TC']
cost_reduction_factor = 0.7
n_lev.weight_network(cycleway_reduc_factor=cost_reduction_factor,grades=True)
n_lev.sample_trips(spec_route=122)
n_lev.get_personal_spatial_features(1,1)
n_lev.add_edge_bearing()

C:\Users\latitude\AppData\Local\Temp\ipykernel_24876\1063141680.py:459: RuntimeWarning: invalid value encountered in divide
  age_multiplier/=np.max(age_multiplier)


In [97]:
B=0
name = input('Filename:')
date_time_str = datetime.now().strftime('%m-%d')
icp = 0
ltp = 0
rtp = 0
tsp = 0
proximity_dist = 10
beta_age = 1
mu_age = 1
scale_age = 1
beta_bikeway = 1
beta_transit = 1
alpha = 0.2
n_lev.run_algo2(budget = B,savepath=f'Figures/reseaux/{name}-{date_time_str}',
            buffer = 500,icp = icp, ltp = ltp,rtp = rtp,tsp = tsp,
            cost_reduction_factor = cost_reduction_factor,proximity_dist=proximity_dist,beta_age = beta_age,mu_age = mu_age,scale_age = scale_age, 
            beta_bikeway = beta_bikeway,
            beta_transit = beta_transit,
            alpha = alpha,
            build_dmin = 0,
            backup_every=10)

Filename: levis_init


Initializing, computing desire boxes...
Dominating angle: 48.464982006376054
Computing routes...


  0%|          | 0/3839 [00:00<?, ?it/s]

31.310237040896073 % of solved routes,  0.20838760093774422  % of static routes
Computing route metrics...
Network backup at Figures/reseaux/levis_init-10-21


C:\Users\latitude\AppData\Local\Temp\ipykernel_24876\1063141680.py:392: UserWarning: The GeoSeries you are attempting to plot is composed of empty geometries. Nothing has been displayed.
  new_network = built_edges.explore(tiles = tiles,


length = 0.0 km


In [ ]:
n_lev.n_ex_edges.to_file("test.geojson", driver="GeoJSON")

In [ ]:
n_lev.n_ex_edges.explore(column = 'grade_abs',vmin = 0,vmax = 0.1)

In [ ]:
route = n_lev.routes.sample(1)
idx = route.index
m = n_lev.all_routes_edges[n_lev.all_routes_edges.route_number == idx[0]]
display(m.explore())
lengths = np.concatenate(([0], m.length.cumsum().values))

plt.plot(lengths,n_lev.n_pot_nodes.loc[route.nodes.values[0]].elevation.values)

In [146]:
n_ex = ox.io.load_graphml(ex)
n_ex = ox.project_graph(n_ex)
n_ex_nodes,n_ex_edges = ox.graph_to_gdfs(n_ex)
used_nodes = set(n_ex_edges.index.get_level_values('u')) | set(n_ex_edges.index.get_level_values('v'))
n_ex_nodes = n_ex_nodes[n_ex_nodes.index.isin(used_nodes)]
n_ex_edges['build_iter'] = 0
n_ex_og = ox.graph_from_gdfs(n_ex_nodes.copy(),n_ex_edges.copy())